<a href="https://colab.research.google.com/github/hoin1357/MachineLearning/blob/main/3%EC%B0%A8%EA%B3%BC%EC%A0%9C/3%EC%B0%A8_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 서울대공원 방문객 예측 모델 성능 변화 보고서

작성일: 2026-05-13  
검증 구간: 2025-05-01 ~ 2025-07-31  
우선 목표: 실제 혼잡일을 예측 상위 위험일로 잘 올리는 것  
주요 우선 지표: NDCG@20, Recall@20, Precision@20  

## 1. 지표 해석 기준

| 지표 | 의미 | 해석 |
| --- | --- | --- |
| MAE | 실제 방문객 수와 예측 방문객 수의 절대 오차 평균 | 낮을수록 좋음 |
| RMSE | 큰 오차에 더 큰 벌점을 주는 평균 오차 | 낮을수록 좋음 |
| MAPE | 실제값 대비 오차율 평균 | 낮을수록 좋음 |
| R2 | 실제 변동성을 모델이 설명하는 정도 | 높을수록 좋음 |
| Precision@20 | 예측 상위 20일 중 실제 혼잡일 비율 | 높을수록 좋음 |
| Recall@20 | 실제 혼잡일 중 예측 상위 20일에 잡힌 비율 | 높을수록 좋음 |
| NDCG@20 | 혼잡한 날을 예측 순위 상단에 잘 배치했는지 | 높을수록 좋음 |

이 프로젝트는 단순 평균 오차보다 "혼잡일을 놓치지 않는 것"이 중요하므로, 튜닝 기준은 주로 NDCG@20과 Recall@20을 우선했다.

## 2. 전체 성능 변화 요약

| 단계 | 설명 | MAE | RMSE | MAPE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 기준 모델 | 하이퍼파라미터 튜닝 전 기준 | 3266.38 | 4620.77 | 176.39 | 0.7312 | 0.55 | 0.7857 | 0.8825 |
| 코로나 가중치 실험 | 2020~2022 데이터 가중치 0.25 | 3160.85 | 4376.30 | 185.10 | 0.7589 | 0.55 | 0.7857 | 0.8942 |
| 코로나 반사실+가중치 실험 | 코로나가 없었다고 보정한 값과 원자료 병행 | 3247.17 | 4376.91 | 191.37 | 0.7588 | 0.60 | 0.8571 | 0.9071 |
| 1단계 혼잡일 분류 튜닝 후 | 혼잡일 분류 모델 100회 탐색 결과 | 3160.34 | 4419.26 | 190.01 | 0.7541 | 0.65 | 0.9286 | 0.9399 |
| 2단계 최종 방문객 예측 튜닝 후 | 최종 방문객 수 회귀 모델 100회 탐색 결과 | 3214.38 | 4509.65 | 194.84 | 0.7440 | 0.65 | 0.9286 | 0.9478 |
| 미래 안전 모델 | 실제 미래 예측에 확보 가능한 feature만 사용 | 3347.89 | 5414.60 | 179.34 | 0.6309 | 0.55 | 0.7857 | 0.8292 |
| 참고: 당일 대공원역 데이터 직접 투입 | 실제 서비스에는 사용할 수 없는 분석용 모델 | 2497.10 | 4069.31 | 135.44 | 0.7915 | 0.65 | 0.9286 | 0.9418 |

## 3. 1단계 혼잡일 분류 모델 튜닝

### 최종 선택 파라미터

```python
CLASSIFIER_PARAMS = {
    "loss_function": "Logloss",
    "depth": 6,
    "learning_rate": 0.06,
    "iterations": 800,
    "l2_leaf_reg": 5.0,
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}
```

### 성능 변화

| 비교 | MAE | RMSE | R2 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: |
| 기준 모델 | 3266.38 | 4620.77 | 0.7312 | 0.8825 |
| RMSE 우선 탐색 최고 | 2839.21 | 4188.74 | 0.7791 | 0.8952 |
| 혼잡도 우선 100회 탐색 최고 | 3160.34 | 4419.26 | 0.7541 | 0.9399 |

RMSE 우선 탐색은 평균 오차가 더 작았지만, 혼잡일 순위 성능은 낮았다. 최종 목적이 혼잡일 탐지였기 때문에 NDCG@20이 높은 설정을 선택했다.

## 4. 결측값 보정 모델 튜닝

### 최종 선택 파라미터

```python
IMPUTATION_MODEL_PARAMS = {
    "loss_function": "RMSE",
    "depth": 6,
    "learning_rate": 0.05,
    "iterations": 700,
    "l2_leaf_reg": 5.0,
    "min_data_in_leaf": 5,
    "random_strength": 1.0,
    "verbose": False,
    "random_seed": 42,
    "allow_writing_files": False,
}
```

### 성능 변화

| 단계 | MAE | RMSE | R2 | NDCG@20 | 판단 |
| --- | ---: | ---: | ---: | ---: | --- |
| 기존 결측값 보정 설정 | 3225.01 | 4313.09 | 0.7658 | 0.9300 | 최종 유지 |
| 최고 후보 | 3225.01 | 4313.09 | 0.7658 | 0.9300 | 기존과 동률 |

결측값 보정 모델은 파라미터를 바꿔도 전체 성능 개선이 없었다. 따라서 기존 설정을 유지했다.

## 5. 2단계 최종 방문객 수 예측 모델 튜닝

### 최종 선택 파라미터

```python
REGRESSOR_PARAMS = {
    "loss_function": "RMSE",
    "depth": 10,
    "learning_rate": 0.04,
    "iterations": 1200,
    "l2_leaf_reg": 8.0,
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}
```

### 성능 변화

| 단계 | MAE | RMSE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 1단계 튜닝 후 | 3160.34 | 4419.26 | 0.7541 | 0.65 | 0.9286 | 0.9399 |
| 2단계 최종 튜닝 후 | 3214.38 | 4509.65 | 0.7440 | 0.65 | 0.9286 | 0.9478 |

2단계 튜닝 후 MAE와 RMSE는 약간 나빠졌지만, NDCG@20은 0.9399에서 0.9478로 상승했다.

## 6. 대공원역 방문 데이터 추가 조사

### 방문객 수와 대공원역 승하차의 상관관계

| 구간 | N | Pearson | Spearman | 평균 방문객 | 평균 승하차 |
| --- | ---: | ---: | ---: | ---: | ---: |
| 전체 2015~2025.07 | 3762 | 0.8229 | 0.8247 | 5360.6 | 9926.0 |
| 코로나 전 2015~2019 | 1723 | 0.8636 | 0.8452 | 6854.1 | 12172.0 |
| 코로나 2020~2022 | 1096 | 0.7452 | 0.7834 | 3785.0 | 6782.3 |
| 코로나 후 2023~2025.07 | 943 | 0.7402 | 0.8483 | 4462.7 | 9476.1 |
| 방문객 상위 15% | 565 | 0.6697 | 0.5577 | 20094.5 | 21645.2 |

대공원역 당일 승하차 데이터는 방문객 수와 매우 강한 관계가 있었다. 하지만 미래 예측 시점에는 예측 대상 날짜의 실제 대공원역 승하차를 알 수 없으므로 최종 서비스 모델에는 직접 넣지 않았다.

### 지나간 교통량과 날씨의 연관도 비교

| 데이터 | 대표 변수 | Spearman |
| --- | --- | ---: |
| 당일 교통량 | 당일 대공원역 하차 | 0.8253 |
| 과거 교통량 | 1일 전 대공원역 하차 | 0.6216 |
| 과거 교통량 | 7일 전 대공원역 하차 | 0.6187 |
| 날씨 | 18도에서 떨어진 정도 | -0.5939 |
| 날씨 | 너무 추운 날 여부 | -0.4501 |
| 날씨 | 일강수량 | -0.2259 |

## 7. 대공원역 데이터를 넣었을 때의 성능

| 모델 | MAE | RMSE | MAPE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 2단계 최종 튜닝 모델 | 3214.38 | 4509.65 | 194.84 | 0.7440 | 0.65 | 0.9286 | 0.9478 |
| 당일 대공원역 데이터 직접 투입 | 2497.10 | 4069.31 | 135.44 | 0.7915 | 0.65 | 0.9286 | 0.9418 |

방문객 수 오차는 크게 줄었지만, 이는 예측 대상 날짜의 실제 교통량을 알고 있다고 가정한 결과다. 실제 운영에서는 불가능하므로 참고용 성능으로만 남겼다.

## 8. 실제 미래 예측용 모델로 변경

### 최종 서비스 모델 feature

```text
날짜
요일
주말 여부
공휴일 여부
연휴 여부
방학/성수기 여부
행사 여부
날씨 API 예보 평균기온
날씨 API 예보 강수량
강수 여부
폭우 여부
너무 추움 여부
너무 더움 여부
```

### 성능 변화

| 모델 | MAE | RMSE | MAPE | R2 | Precision@20 | Recall@20 | NDCG@20 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 2단계 최종 튜닝 모델 | 3214.38 | 4509.65 | 194.84 | 0.7440 | 0.65 | 0.9286 | 0.9478 |
| 실제 미래 예측용 모델 | 3347.89 | 5414.60 | 179.34 | 0.6309 | 0.55 | 0.7857 | 0.8292 |

성능은 하락했다. 하지만 이 결과가 실제 서비스에서 기대할 수 있는 성능에 더 가깝다. 기존 모델은 성능이 좋아 보였지만, 장기 미래 예측에서는 예측값을 다시 입력으로 쓰는 구조라 운영 안정성이 낮았다.

## 9. 최종 판단

### 성능만 보면

```text
당일 대공원역 데이터 직접 투입 모델
> 2단계 하이퍼파라미터 튜닝 모델
> 실제 미래 예측용 모델
```

### 실제 서비스 가능성까지 고려하면

```text
실제 미래 예측용 모델
> 2단계 하이퍼파라미터 튜닝 모델
> 당일 대공원역 데이터 직접 투입 모델
```

현재 프로젝트 조건에서는 최종 장기 예측 모델을 날짜/행사/날씨 중심으로 운영하는 것이 가장 안전하다.

## 10. 생성/참고 산출물

| 산출물 | 내용 |
| --- | --- |
| `backend/data/artifacts/classifier_random100_summary.md` | 1단계 혼잡일 분류 모델 100회 탐색 기록 |
| `backend/data/artifacts/imputation_tuning_summary.md` | 결측값 보정 모델 튜닝 기록 |
| `backend/data/artifacts/regressor_random100_summary.md` | 2단계 방문객 수 예측 모델 100회 탐색 기록 |
| `backend/data/artifacts/covid_preprocessing_model_comparison.csv` | 코로나 전처리 실험 비교 |
| `backend/data/artifacts/external_transport/daegongwon_subway_correlation_summary.csv` | 대공원역 승하차와 방문객 상관관계 |
| `backend/data/artifacts/external_transport/traffic_vs_weather_correlation.csv` | 교통 데이터와 날씨 데이터 연관도 비교 |
| `backend/data/artifacts/portable_predictions.csv` | 최종 실제 미래 예측용 캐시 |
